# 1.1 Foundational Models with LangChain

## What you will learn in this notebook

- How to **initialise and invoke a language model** using LangChain
- How to **customise model behaviour** (e.g. temperature)
- How to **switch between model providers** (OpenAI, Anthropic, Google)
- What an **Agent** is and why it is different from a plain model call
- How to **stream output** token by token

---

### Key vocabulary before we start

| Term | Plain English |
|---|---|
| **LLM / Foundation Model** | A large neural network pre-trained on vast text data. It takes text in and returns text out. Examples: GPT-5-nano, Claude Sonnet, Gemini Flash. |
| **LangChain** | A Python library that gives us a *common interface* to talk to many different LLM providers without rewriting our code each time. |
| **Invoke** | The simplest way to call a model — send a message, wait for the full reply to come back. |
| **Agent** | A model that can **reason and take actions** (e.g. call tools, search the web, run code) rather than just reply once. Think of it as a model that can loop: think → act → observe → think again. |
| **Streaming** | Getting the model's reply **word by word** as it is generated, rather than waiting for the whole response. |
| **Temperature** | A knob (0.0 – 2.0) that controls how creative/random the model is. 0 = very deterministic, 1+ = more creative. |

In [2]:
# ============================================================
# CELL 1: Load Environment Variables
# ============================================================
# We store API keys (OpenAI, Anthropic, Google etc.) in a .env file
# so they are NEVER hard-coded in our notebooks or committed to Git.
#
# The .env file lives in the project root and looks like this:
#   OPENAI_API_KEY=sk-...
#   ANTHROPIC_API_KEY=sk-ant-...
#   GOOGLE_API_KEY=AI...
#
# load_dotenv() reads that file and injects those values into
# os.environ so that LangChain (and any other library) can find them
# automatically without us passing the key explicitly.
#
# Returns True if the .env file was found and loaded successfully.

from dotenv import load_dotenv

load_dotenv("/Users/cheranreddy/PycharmProjects/Learning_Agentic_AI/05-rag/.env")

True

In [3]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

key = os.getenv("OPENAI_API_KEY")

print("Loaded:", key is not None)
print("Prefix:", key[:10] if key else None)

Loaded: True
Prefix: sk-proj-vf


---
## Section 1 — Initialising and Invoking a Model

### How does `init_chat_model` work?

LangChain's `init_chat_model` is a **factory function** — you pass in a model name and it figures out which provider to use automatically (OpenAI, Anthropic, Google, etc.).

Behind the scenes it:
1. Detects the provider from the model name (e.g. `gpt-*` → OpenAI, `claude-*` → Anthropic)
2. Reads the relevant API key from your environment
3. Returns a **chat model object** that has a consistent interface (`.invoke()`, `.stream()`, etc.)

This means you can **swap providers by changing one string** — the rest of your code stays the same.

In [4]:
# ============================================================
# CELL 2: Initialise a Chat Model
# ============================================================
# init_chat_model is LangChain's universal constructor for LLMs.
#
# - model="gpt-5-nano"  → tells LangChain to use OpenAI's gpt-5-nano
#   (the "nano" variant is fast and cheap — good for learning)
#
# At this point NO API call is made. We are just configuring the object.
# The actual HTTP request happens only when we call .invoke() below.

from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-5.4-mini",model_provider="openai")

In [5]:
# ============================================================
# CELL 3: Invoke the Model (Blocking Call)
# ============================================================
# .invoke() sends our message to the model and WAITS for the
# full response before returning. It is the simplest way to
# call a model — one shot in, one shot out.
#
# You can pass:
#   - A plain string  (LangChain wraps it in a HumanMessage for you)
#   - A list of message objects (for multi-turn conversations)
#
# The return value is an AIMessage object, not just a string.
# AIMessage has several fields:
#   .content          → the actual text reply  ← what you usually want
#   .response_metadata → token counts, model name, finish reason, etc.
#   .tool_calls        → any tool/function calls the model wants to make

response = model.invoke("What's the capital of the Moon?")

response   # Shows the full AIMessage object

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


AIMessage(content='The Moon doesn’t have a capital — it has no countries or cities.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 13, 'total_tokens': 31, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5.4-mini-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EIGs5nhcTQqtOhSVHn29qD9lEXs9e', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a04e80-d6e1-7a11-8487-fcce423bb547-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 18, 'total_tokens': 31, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTP

Loaded: True
Prefix: sk-proj-oN


In [6]:
# ============================================================
# CELL 4: Extract the Text Reply
# ============================================================
# response.content is the plain text string we care about most.
# In real applications this is what you would display to the user
# or pass to the next step of your pipeline.

print(response.content)

The Moon doesn’t have a capital — it has no countries or cities.


In [19]:
response.response_metadata

{'token_usage': {'completion_tokens': 41,
  'prompt_tokens': 13,
  'total_tokens': 54,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 0,
   'rejected_prediction_tokens': 0},
  'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}},
 'model_name': 'gpt-5.4-mini-2026-03-17',
 'system_fingerprint': None,
 'id': 'chatcmpl-EICKqmiBDySOYTrjGvFWdY1XWFNzW',
 'service_tier': 'default',
 'finish_reason': 'stop',
 'logprobs': None}

In [20]:
# ============================================================
# CELL 5: Inspect Token Usage and Metadata
# ============================================================
# response_metadata is returned by every LLM call and is extremely
# useful for:
#
#   'token_usage'      → how many tokens were consumed (= your cost!)
#       'prompt_tokens'    → tokens in your input message
#       'completion_tokens'→ tokens the model generated in its reply
#       'total_tokens'     → prompt + completion
#       'reasoning_tokens' → tokens spent on internal chain-of-thought
#                            (only present for "thinking" models like o1/gpt-5-nano)
#
#   'model_name'       → the exact model version that answered
#   'finish_reason'    → why the model stopped:
#                          'stop'   = normal end
#                          'length' = hit max_tokens limit
#                          'tool_calls' = model wants to use a tool
#
# 💡 Rule of thumb: always log token usage in production so you can
#    track costs and catch runaway prompts early.

from pprint import pprint

pprint(response.response_metadata)

{'finish_reason': 'stop',
 'id': 'chatcmpl-EICKqmiBDySOYTrjGvFWdY1XWFNzW',
 'logprobs': None,
 'model_name': 'gpt-5.4-mini-2026-03-17',
 'service_tier': 'default',
 'system_fingerprint': None,
 'token_usage': {'completion_tokens': 41,
                 'completion_tokens_details': {'accepted_prediction_tokens': 0,
                                               'audio_tokens': 0,
                                               'reasoning_tokens': 0,
                                               'rejected_prediction_tokens': 0},
                 'prompt_tokens': 13,
                 'prompt_tokens_details': {'audio_tokens': 0,
                                           'cached_tokens': 0},
                 'total_tokens': 54}}


---
## Section 2 — Customising Your Model

### What is Temperature?

When a model generates the next token (word piece), it calculates a probability score for every possible token in its vocabulary. **Temperature scales those probabilities** before sampling:

- **temperature = 0.0** → Always picks the single highest-probability token. Completely deterministic. Same input → same output every time. Good for: code generation, data extraction, factual Q&A.
- **temperature = 1.0** → Samples proportionally to the raw probabilities. Balanced. Good for: general chat, customer support.
- **temperature > 1.0** → Amplifies unlikely tokens, making output more random and creative. Good for: brainstorming, creative writing.

> 💡 **When to use what:** If your task has one correct answer → low temperature. If you want variety and creativity → higher temperature.

In [7]:
# ============================================================
# CELL 6: Customise the Model with Parameters
# ============================================================
# Any keyword argument you pass to init_chat_model beyond 'model'
# is forwarded directly to the underlying provider's API.
#
# Common parameters (most providers support these):
#
#   temperature   (float, 0.0–2.0)
#       Controls randomness. 0 = deterministic, 1+ = creative.
#
#   max_tokens    (int)
#       Hard cap on the number of tokens the model can generate.
#       Use this to control costs and response length.
#
#   top_p         (float, 0.0–1.0)
#       Nucleus sampling — only consider the top-p% of probability
#       mass when picking the next token. Alternative to temperature.
#
# Here we set temperature=1.0 for a slightly more varied response.
# Notice the answer below is worded differently from Cell 3,
# even though the question is identical — that's temperature at work.

model = init_chat_model(
    model="gpt-5-nano",
    temperature=1.0      # More creative / varied replies
)

response = model.invoke("What's the capital of the Moon?")
print(response.content)

There isn’t one. The Moon is not a sovereign country and has no government or capital. It’s governed and explored under international agreements (like the Outer Space Treaty and Artemis Accords) and by the space agencies that operate there.

If you’re thinking of a fictional or future scenario, I can suggest some ideas for a lunar capital and its location. Want that?


---
## Section 3 — Switching Model Providers

One of LangChain's biggest strengths is **provider abstraction**. The `.invoke()` / `.stream()` interface is identical regardless of whether you use OpenAI, Anthropic, or Google — only the model name string changes.

### Provider detection rules
- `gpt-*` or `o1-*` → **OpenAI** (needs `OPENAI_API_KEY`)
- `claude-*` → **Anthropic** (needs `ANTHROPIC_API_KEY`)
- Pass a `ChatGoogleGenerativeAI` object directly → **Google** (needs `GOOGLE_API_KEY`)

Full list of integrations: https://docs.langchain.com/oss/python/integrations/chat

In [8]:
# ============================================================
# CELL 7: Using Anthropic's Claude via LangChain
# ============================================================
# Just change the model string — everything else stays the same.
# LangChain detects "claude-" prefix and routes to Anthropic.
#
# Prerequisite: ANTHROPIC_API_KEY must be set in your .env file.
#
# claude-sonnet-4-5 is Anthropic's balanced model:
#   - Stronger reasoning than Haiku
#   - Faster and cheaper than Opus
#   - Great for most applied GenAI use cases

model = init_chat_model(model="claude-sonnet-4-5")

response = model.invoke("What's the capital of the Moon?")
print(response.content)

TypeError: "Could not resolve authentication method. Expected one of api_key, auth_token, or credentials to be set. Or for one of the `X-Api-Key` or `Authorization` headers to be explicitly omitted"

In [ ]:
# ============================================================
# CELL 8: Using Google Gemini via a Native LangChain Integration
# ============================================================
# Google's models need the langchain_google_genai package.
# Unlike OpenAI/Anthropic, we instantiate ChatGoogleGenerativeAI
# directly instead of using init_chat_model — but once created,
# the object behaves identically (same .invoke(), .stream() API).
#
# Prerequisite: GOOGLE_API_KEY must be set in your .env file.
#   Install: pip install langchain-google-genai
#
# gemini-2.5-flash-lite → Google's fast, low-cost model.
# Good for high-throughput tasks where speed matters.

from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

response = model.invoke("What's the capital of the Moon?")
print(response.content)

---
## Section 4 — What is an Agent?

### Model vs Agent — the key difference

```
PLAIN MODEL CALL
────────────────
  User message  ──→  LLM  ──→  Single reply
  (one round trip, no tools, no memory, no decisions)
```
![image.png](attachment:78f2dc5c-7381-4ea7-8234-aad69bd76e86.png)

An **Agent** = LLM + Tools + a Loop.

- The **LLM** is the brain: it reads the conversation and decides what to do next.
- The **Tools** are actions the agent can take (web search, run Python, query a database, call an API, etc.).
- The **Loop** keeps running until the LLM produces a final answer.

### Why does this matter?

A plain model call is like asking a question to someone locked in a room with no internet, no calculator, and no memory — they can only use what they were taught during training.

An agent is like asking that same person but now they have a smartphone, a notebook, and can send messages to other specialists. Much more powerful.

### The `create_agent` function

`create_agent` in LangChain wraps the LLM inside a **ReAct** (Reason + Act) loop powered by LangGraph under the hood. You get:
- Conversation memory (a `messages` state list)
- Tool-calling capability (add tools when you need them)
- Streaming support out of the box

In [24]:
%pip install -U langchain

  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached ormsgpack-1.12.2-cp313-cp313-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (3.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.5/571.5 kB 1.8 MB/s  0:00:0036m-:--:--
Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl (41 kB)
Using cached ormsgpack-1.12.2-cp313-cp313-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl (378 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.86
    Uninstalling langchain-core-0.3.86:
      Successfully uninstalled langchain-core-0.3.86
  Attempting uninstall: langchain━━━━━╺━━━━━━━━━ 6/8 [langgraph]sdk]]
    Found existing installation: langchain 0.3.30━━━━━━━━━ 6/8 [langgraph]
    Uninstalling langchain-0.3.30:╺━━━━━━━━━ 6/8 [langgraph]
      Successfully uninstalled langchain-0.3.30m━━━━━━━━━ 6/8 [langgraph]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [langchain]/8 [

In [9]:
# ============================================================
# CELL 9: Create an Agent from an Existing Model Object
# ============================================================
# create_agent wraps any LangChain chat model in an agent loop.
#
# Option A (this cell): pass the model object you already have.
# This is useful when you want to reuse a configured model
# (e.g. one that already has temperature and tools set up).
#
# The agent's state is stored in a 'messages' key — a list that
# accumulates the full conversation history (HumanMessages,
# AIMessages, ToolMessages, etc.).

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent

model = init_chat_model(model="gpt-5.4-mini",model_provider="openai")
agent = create_agent(model=model)   # Pass the model object directly

In [23]:
# ============================================================
# CELL 10: Create an Agent from a Model Name String
# ============================================================
# Option B: pass a model name string — create_agent calls
# init_chat_model internally. Shorter syntax, same result.

agent_claude = create_agent(model="claude-sonnet-4-5")

NameError: name 'create_agent' is not defined

In [10]:
# ============================================================
# CELL 11: Create an Agent — Positional String Shorthand
# ============================================================
# Option C: pass the model name as a positional argument.
# Cleanest syntax for quick prototyping.
from langchain.agents import create_agent
agent = create_agent("gpt-5-nano")

In [11]:
# ============================================================
# CELL 12: Invoke the Agent
# ============================================================
# Unlike model.invoke("some string"), agent.invoke() expects a
# DICTIONARY with a 'messages' key.
#
# Why a dictionary and not just a string?
#   Because agents manage STATE. The state is the full conversation
#   history. Passing a dict lets LangGraph track and update that
#   state as the agent reasons through the problem.
#
# HumanMessage wraps the user's text so LangChain knows the role
# (Human vs AI vs Tool) for proper prompt formatting.
#
# The agent will:
#   1. Send the message to the LLM
#   2. Check if the LLM wants to use any tools
#   3. If yes → run the tool, feed results back to LLM, repeat
#   4. If no  → return the final AIMessage

from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages":
         [
             HumanMessage(content="What's the capital of the Moon?")
         ]
    }
)

In [12]:
# ============================================================
# CELL 13: Inspect the Agent's Full Response
# ============================================================
# The agent returns the COMPLETE state dict, not just the reply.
# response['messages'] is a list of ALL messages in the conversation:
#   [HumanMessage, AIMessage]  (for a simple single-turn exchange)
# or for a multi-step agent:
#   [HumanMessage, AIMessage(tool_call), ToolMessage, AIMessage(final)]
#
# Inspecting this is very useful for debugging — you can see exactly
# what the agent decided to do and why.

from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content="What's the capital of the Moon?", additional_kwargs={}, response_metadata={}, id='30b900b0-599b-472d-bcd6-b0716f09534c'),
              AIMessage(content='There isn’t one. The Moon has no government or capital city—just the natural satellite. In fiction people sometimes invent a lunar capital (e.g., Lunaris, Selene City, Mareopolis), but these are purely fictional. If you want, I can come up with a few more ideas for a sci‑fi setting.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 844, 'prompt_tokens': 13, 'total_tokens': 857, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EIGtyGah7xfDaiYrmu2mvjfq8krF8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprob

In [13]:
init_new_interaction = HumanMessage(content="Can give me fictional captial?")
response['messages'].append(init_new_interaction)
response

{'messages': [HumanMessage(content="What's the capital of the Moon?", additional_kwargs={}, response_metadata={}, id='30b900b0-599b-472d-bcd6-b0716f09534c'),
  AIMessage(content='There isn’t one. The Moon has no government or capital city—just the natural satellite. In fiction people sometimes invent a lunar capital (e.g., Lunaris, Selene City, Mareopolis), but these are purely fictional. If you want, I can come up with a few more ideas for a sci‑fi setting.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 844, 'prompt_tokens': 13, 'total_tokens': 857, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EIGtyGah7xfDaiYrmu2mvjfq8krF8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, i

In [14]:
response  = agent.invoke(response)
pprint(response)

{'messages': [HumanMessage(content="What's the capital of the Moon?", additional_kwargs={}, response_metadata={}, id='30b900b0-599b-472d-bcd6-b0716f09534c'),
              AIMessage(content='There isn’t one. The Moon has no government or capital city—just the natural satellite. In fiction people sometimes invent a lunar capital (e.g., Lunaris, Selene City, Mareopolis), but these are purely fictional. If you want, I can come up with a few more ideas for a sci‑fi setting.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 844, 'prompt_tokens': 13, 'total_tokens': 857, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EIGtyGah7xfDaiYrmu2mvjfq8krF8', 'service_tier': 'default', 'finish_reason': 'stop', 'logprob

In [15]:
# ============================================================
# CELL 14: Extract Just the Final Reply
# ============================================================
# response['messages'] is the full message history list.
# The LAST message [-1] is always the agent's final answer.
# .content gives us the plain text string.
#
# Pattern to memorise for agents:
#   response['messages'][-1].content  →  final answer text

print(response['messages'][-1].content)

Sure! Here are some fictional Moon capitals you can drop into a sci‑fi setting. If you want, I can flesh out one into a full write‑up with districts, leaders, and a map.

- Lunaris Prime — The political heart of a Lunar Confederation, built in a shielded equatorial crater with vast glass domes and a web of transit tubes. Government is parliamentary with sectoral councils.

- Selene City — Capital of a Moon Commonwealth; a cosmopolitan ring city around a mare, fed by orbital elevators and trade with Earth and Mars. Culture is artistic and multilingual.

- Mareopolis — A sprawling metropolis that straddles Mare Serenitatis and nearby plains, carved into basalt and ice caverns; tech‑driven governance with a technocrat flavor.

- Tycho Citadel — A fortress‑city perched on the Tycho crater rim; strong military and research presence, with a ceremonial capital district and a strongman/commander‑in‑chief vibe.

- Polaris Nexus — Polar capital near Shackleton Crater; centers on ice mining, cryo

In [16]:
# ============================================================
# CELL 15: Multi-Turn Conversation with an Agent
# ============================================================
# Agents can maintain context across multiple turns.
# We simulate a conversation by passing the full history:
#
#   Turn 1: Human asks about the Moon's capital
#   Turn 2: AI (we're pretending) says "Luna City"
#   Turn 3: Human asks for more details about Luna City
#
# By including the prior Human + AI messages, the agent understands
# "Luna City" refers to the fictional capital it invented earlier.
#
# 💡 This is called 'conversation history injection'. In a real app
#    you would load this history from a database per user session.
#    LangChain has memory modules that do this automatically.

from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What's the capital of the Moon?"),
        AIMessage(content="The capital of the Moon is Luna City."),  # Prior AI reply
        HumanMessage(content="Interesting, tell me more about Luna City")  # Follow-up
    ]}
)

pprint(response)

{'messages': [HumanMessage(content="What's the capital of the Moon?", additional_kwargs={}, response_metadata={}, id='f2080424-6d43-408a-a5c7-6df15ee1a054'),
              AIMessage(content='The capital of the Moon is Luna City.', additional_kwargs={}, response_metadata={}, id='38b9aec4-53e8-4990-8440-7b2234ff3ecc', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content='Interesting, tell me more about Luna City', additional_kwargs={}, response_metadata={}, id='076c40f2-c3b7-488b-a43d-76de3561a0a0'),
              AIMessage(content='Luna City is a fictional setting, since there’s no real “capital” of the Moon. Here’s a rich, self-contained concept you can use for stories, games, or world-building.\n\n- Location and layout\n  - Nestled in the rim and inner walls of a polar crater (often imagined near Shackleton), Luna City uses the crater as natural shielding from radiation and micrometeorites.\n  - The city climbs from the crater floor into tiered domes and tunnels, 

---
## Section 5 — Streaming Output

### Why stream?

With `.invoke()`, the user stares at a blank screen until the **entire** response is ready — which could be 5–15 seconds for a long answer.

With `.stream()`, tokens appear **progressively** as the model generates them, just like ChatGPT's typing effect. This dramatically improves perceived performance and user experience.

### `stream_mode="messages"`

This tells LangGraph to emit **individual message chunks** (one per token) rather than the full state update. Each chunk has:
- `token.content` — the new text added in this chunk (often a single word or punctuation)
- `metadata` — which node/agent produced this token (useful in multi-agent graphs)

In [ ]:
# ============================================================
# CELL 16: Stream Tokens in Real Time
# ============================================================
# agent.stream() is a Python generator — it yields chunks one by one
# as the model produces them, so we can print each token immediately.
#
# stream_mode="messages"  → yield message chunks (best for displaying text)
# stream_mode="values"    → yield full state snapshots (best for debugging)
# stream_mode="updates"   → yield only what changed (best for complex graphs)
#
# In the loop:
#   token    → a message chunk object (has .content, .type, .tool_calls)
#   metadata → dict with 'langgraph_node' telling you which node fired
#
# We guard with `if token.content` to skip empty chunks that appear
# at the start/end of tool calls or when the model is deciding.
#
# print(..., end="", flush=True)
#   end=""     → no newline between tokens (they appear inline)
#   flush=True → force output to appear immediately (don't buffer)
from langchain.messages import HumanMessage
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="Tell me all about Luna City, the capital of the Moon")]},
    stream_mode="messages"
):
    
    if token.content:                              # Skip empty/control chunks
        print(token.content, end="")   # Print token immediately

---
## Summary — What we covered

| Concept | Key takeaway |
|---|---|
| `init_chat_model` | Universal factory for any LLM provider — swap the model string, keep the rest of your code |
| `.invoke()` | One-shot call — send message, get full reply back |
| `temperature` | 0 = deterministic, 1+ = creative. Match to your task |
| Provider switching | OpenAI, Anthropic, Google all share the same `.invoke()` / `.stream()` API in LangChain |
| Agent | LLM + Tools + Loop. Can reason, act, observe, and decide when it is done |
| Agent state | Always a dict with `messages` key. Final answer is `response['messages'][-1].content` |
| Streaming | Use `.stream(stream_mode="messages")` to print tokens as they arrive — better UX |

### Next steps
- **1.2** — Prompt Engineering: System messages, few-shot examples, COSTAR framework
- **1.3** — Adding Tools to your Agent: web search, calculators, database queries